# Container RL — Test All Actions via Gymnasium

Walks through every action type using `env.step()`, demonstrating the
correct **v3 multi-head action arrays** under the no-op + per-mode
masking design.

## Action space quick reference

Heads (5): `[action_type (12), opponent (np), colour (nc+1), price_slot (11), purchase (32)]`

| Head | Size (4p,5c) | Index 0 | Indices meaning |
|------|-------------|---------|-----------------|
| action_type | 12 | no-op | 1=BuyFactory..11=DomesticSale |
| opponent | 4 | no-op | 1..3 = 1st..3rd clockwise opponent |
| colour | 6 | no-op | 1..5 = colours 0..4 |
| price_slot | 11 | no-op | 1..10 = $1..$10 |
| purchase | 32 | no-op | 1–5=harbour $2-$6, 6–30=auction bids, 31=STOP |

All sub-head values offset +1 for the no-op at index 0.

In [7]:
import jax
import jax.numpy as jnp
import numpy as np

jax.config.update("jax_disable_jit", True)

from container_rl.env.container import (
    ACTION_BUY_FACTORY, ACTION_BUY_FROM_FACTORY_STORE, ACTION_BUY_WAREHOUSE,
    ACTION_PRODUCE, ACTION_MOVE_LOAD, ACTION_MOVE_SEA, ACTION_MOVE_AUCTION,
    ACTION_PASS, ACTION_TAKE_LOAN, ACTION_REPAY_LOAN, ACTION_DOMESTIC_SALE,
    INITIAL_CASH, LOCATION_OPEN_SEA, PURCHASE_STOP, ContainerJaxEnv,
)

P_STOP = PURCHASE_STOP  # 31

## Helpers

In [8]:
def print_state(env, label=""):
    s = env.unwrapped.state
    # s = env.state
    p = int(s.current_player)
    print(f"  [{label}] P{p} | cash={int(s.cash[p])} loans={int(s.loans[p])}"
          f" factories={int(jnp.sum(s.factory_colors[p]))}"
          f" wh={int(s.warehouse_count[p])}"
          f" harbour={int(jnp.sum(s.harbour_store[p]))}"
          f" ship={int(jnp.sum(s.ship_contents[p]>0))}"
          f" shop={int(s.shopping_active)} prod={int(s.produce_active)}"
          f" auction={int(s.auction_active)} acts={int(s.actions_taken)}")

def step_and_print(env, action, label=""):
    arr = np.asarray(action, dtype=np.int32)
    print(f"  → step({label}): {list(arr)}")
    obs, reward, term, trunc, info = env.step(arr)
    print_state(env, f"after {label}")
    return obs, reward, term, trunc, info

def build_action(action_type, color=0, price_slot=0, opponent=0, purchase=P_STOP):
    return jnp.array(
        [action_type + 1, opponent + 1, color + 1, price_slot + 1, purchase],
        dtype=jnp.int32,
    )

def fresh_env():
    env = ContainerJaxEnv(num_players=4, num_colors=5)
    env.reset()
    return env

---
## Action 0 — Buy Factory

Single-step parallel.  Action: `[1, 0, colour+1, 0, P_STOP]`

In [9]:
print("=" * 60)
print("Action 0: Buy Factory")
print("=" * 60)
env = fresh_env()
print_state(env, "initial")
# Buy colour 2.  2nd factory cost: (1+1)*3 = $6
step_and_print(env, build_action(ACTION_BUY_FACTORY, color=2), "buy colour-2")
assert int(jnp.sum(env.state.factory_colors[0])) == 2
assert int(env.state.cash[0]) == INITIAL_CASH - 6
print("  ✓ passed")

Action 0: Buy Factory
  [initial] P0 | cash=20 loans=0 factories=1 wh=1 harbour=0 ship=0 shop=0 prod=0 auction=0 acts=0
  → step(buy colour-2): [np.int32(1), np.int32(1), np.int32(3), np.int32(1), np.int32(31)]
  [after buy colour-2] P0 | cash=14 loans=0 factories=2 wh=1 harbour=0 ship=0 shop=0 prod=0 auction=0 acts=1
  ✓ passed


---
## Action 1 — Buy Warehouse

Single-step.  Action: `[2, 0, 0, 0, P_STOP]`

In [10]:
print("=" * 60)
print("Action 1: Buy Warehouse")
print("=" * 60)
env = fresh_env()
step_and_print(env, build_action(ACTION_BUY_WAREHOUSE), "buy warehouse")
assert int(env.state.warehouse_count[0]) == 2
assert int(env.state.cash[0]) == INITIAL_CASH - 4
print("  ✓ passed")

Action 1: Buy Warehouse
  → step(buy warehouse): [np.int32(2), np.int32(1), np.int32(1), np.int32(1), np.int32(31)]
  [after buy warehouse] P0 | cash=16 loans=0 factories=1 wh=2 harbour=0 ship=0 shop=0 prod=0 auction=0 acts=1
  ✓ passed


---
## Action 2 — Produce

**Recurrent**: enter + per-factory steps.

1. Enter: `[3, 0, 0, 0, P_STOP]` — pays $1 union dues, `produce_active=1`
2. Recurrent: `[0, 0, colour+1, slot+1, P_STOP]` — process one factory.
   Slot 0-3 = $1-$4, slot 4 = leave idle.

In [11]:
print("=" * 60)
print("Action 2: Produce (single factory)")
print("=" * 60)
env = fresh_env()
print_state(env, "initial")
# Enter
step_and_print(env, build_action(ACTION_PRODUCE), "enter")
assert int(env.state.produce_active) == 1
assert int(env.state.cash[0]) == INITIAL_CASH - 1
# Produce colour 0 at $2 (slot 1 → index 2)
step_and_print(env, jnp.array([0, 0, 1, 2, P_STOP], dtype=jnp.int32),
               "colour 0 @ $2")
assert int(env.state.produce_active) == 0  # batch done
print("  ✓ passed")

Action 2: Produce (single factory)
  [initial] P0 | cash=20 loans=0 factories=1 wh=1 harbour=0 ship=0 shop=0 prod=0 auction=0 acts=0
  → step(enter): [np.int32(3), np.int32(1), np.int32(1), np.int32(1), np.int32(31)]
  [after enter] P0 | cash=19 loans=0 factories=1 wh=1 harbour=0 ship=0 shop=0 prod=1 auction=0 acts=0
  → step(colour 0 @ $2): [np.int32(0), np.int32(0), np.int32(1), np.int32(2), np.int32(31)]
  [after colour 0 @ $2] P0 | cash=19 loans=0 factories=1 wh=1 harbour=0 ship=0 shop=0 prod=0 auction=0 acts=1
  ✓ passed


### Produce — multiple factories + leave idle

In [12]:
env = fresh_env()
# Give P0 a second factory (colour 3)
st = env.state
env.state = st._replace(
    factory_colors=st.factory_colors.at[0, 3].set(1),
    factory_store=st.factory_store.at[0, 3, 1].set(1),
)
print_state(env, "2 factories")
step_and_print(env, build_action(ACTION_PRODUCE), "enter")
# Factory 1: colour 0 at $2
step_and_print(env, jnp.array([0, 0, 1, 2, P_STOP], dtype=jnp.int32), "colour 0 @ $2")
assert int(env.state.produce_active) == 1  # colour 3 pending
# Factory 2: colour 3 leave idle (slot 4 → index 5)
step_and_print(env, jnp.array([0, 0, 4, 5, P_STOP], dtype=jnp.int32), "colour 3 idle")
assert int(env.state.produce_active) == 0
print("  ✓ passed")

  [2 factories] P0 | cash=20 loans=0 factories=2 wh=1 harbour=0 ship=0 shop=0 prod=0 auction=0 acts=0
  → step(enter): [np.int32(3), np.int32(1), np.int32(1), np.int32(1), np.int32(31)]
  [after enter] P0 | cash=19 loans=0 factories=2 wh=1 harbour=0 ship=0 shop=0 prod=1 auction=0 acts=0
  → step(colour 0 @ $2): [np.int32(0), np.int32(0), np.int32(1), np.int32(2), np.int32(31)]
  [after colour 0 @ $2] P0 | cash=19 loans=0 factories=2 wh=1 harbour=0 ship=0 shop=0 prod=1 auction=0 acts=0
  → step(colour 3 idle): [np.int32(0), np.int32(0), np.int32(4), np.int32(5), np.int32(31)]
  [after colour 3 idle] P0 | cash=19 loans=0 factories=2 wh=1 harbour=0 ship=0 shop=0 prod=0 auction=0 acts=1
  ✓ passed


---
## Action 3 — Buy from Factory Store

**Recurrent**: opponent selection + colour/harbour-price loop + STOP.

1. Step 1: `[4, opp+1, 0, 0, P_STOP]` — enters `shopping_active=1`
2. Step 2+: `[0, 0, colour+1, 0, harbour_idx]` — harbour_idx 1-5 = $2-$6
3. STOP: `[0, 0, 0, 0, P_STOP]`

Environment auto-selects the cheapest source slot for the chosen colour.

In [13]:
print("=" * 60)
print("Action 3: Buy from Factory Store")
print("=" * 60)
env = fresh_env()
print_state(env, "initial (P1 has colour 1 @ slot 4)")
# Step 1: opponent
step_and_print(env, build_action(ACTION_BUY_FROM_FACTORY_STORE, opponent=0),
               "select P1")
assert int(env.state.shopping_active) == 1
# Step 2: colour 1 (idx 2), harbour $4 (idx 3)
step_and_print(env, jnp.array([0, 0, 2, 0, 3], dtype=jnp.int32),
               "buy colour 1, harbour $4")
assert int(env.state.cash[0]) == INITIAL_CASH - 5  # source $5
# STOP
step_and_print(env, jnp.array([0, 0, 0, 0, P_STOP], dtype=jnp.int32), "STOP")
assert int(env.state.shopping_active) == 0
print("  ✓ passed")

Action 3: Buy from Factory Store
  [initial (P1 has colour 1 @ slot 4)] P0 | cash=20 loans=0 factories=1 wh=1 harbour=0 ship=0 shop=0 prod=0 auction=0 acts=0
  → step(select P1): [np.int32(4), np.int32(1), np.int32(1), np.int32(1), np.int32(31)]
  [after select P1] P0 | cash=20 loans=0 factories=1 wh=1 harbour=0 ship=0 shop=1 prod=0 auction=0 acts=0
  → step(buy colour 1, harbour $4): [np.int32(0), np.int32(0), np.int32(2), np.int32(0), np.int32(3)]
  [after buy colour 1, harbour $4] P0 | cash=18 loans=0 factories=1 wh=1 harbour=1 ship=0 shop=0 prod=0 auction=0 acts=1


AssertionError: 

---
## Action 4 — Move to Harbour + Load

Same structure as Action 3 but containers go to **ship**.

1. Step 1: `[5, opp+1, 0, 0, P_STOP]`
2. Step 2+: `[0, 0, colour+1, 0, 1]` (1 = buy signal)
3. STOP: `[0, 0, 0, 0, P_STOP]`

In [ ]:
print("=" * 60)
print("Action 4: Move to Harbour + Load")
print("=" * 60)
env = fresh_env()
# Give P1 harbour stock: colour 2 at slot 3 ($4)
st = env.state
env.state = st._replace(
    harbour_store=st.harbour_store.at[1, 2, 3].set(1))
print_state(env, "P1 has harbour stock")
# Step 1: select opponent
step_and_print(env, build_action(ACTION_MOVE_LOAD, opponent=0), "select P1")
assert int(env.state.shopping_active) == 1
# Step 2: colour 2 (idx 3), buy (idx 1)
step_and_print(env, jnp.array([0, 0, 3, 0, 1], dtype=jnp.int32), "load colour 2")
assert int(jnp.sum(env.state.ship_contents[0] > 0)) == 1
# STOP
step_and_print(env, jnp.array([0, 0, 0, 0, P_STOP], dtype=jnp.int32), "STOP")
print("  ✓ passed")

---
## Action 5 — Move to Open Sea

Single-step.  Action: `[6, 0, 0, 0, P_STOP]`

In [14]:
print("=" * 60)
print("Action 5: Move to Open Sea")
print("=" * 60)
env = fresh_env()
st = env.state
env.state = st._replace(ship_location=st.ship_location.at[0].set(3))
step_and_print(env, build_action(ACTION_MOVE_SEA), "move to sea")
assert int(env.state.ship_location[0]) == LOCATION_OPEN_SEA
print("  ✓ passed")

Action 5: Move to Open Sea
  → step(move to sea): [np.int32(6), np.int32(1), np.int32(1), np.int32(1), np.int32(31)]
  [after move to sea] P0 | cash=20 loans=0 factories=1 wh=1 harbour=0 ship=0 shop=0 prod=0 auction=0 acts=1
  ✓ passed


---
## Action 6 — Auction

**Recurrent** multi-player.  Auction ends the turn.

1. Initiate: `[7, 0, 0, 0, P_STOP]` (ship at sea, has cargo)
2. Bids: `[7, player_id, 0, 0, bid_amount]` — opponent head = direct player index
3. Seller: `[7, 0, 0, 0, 0]` reject / `[7, 0, 0, 0, 1]` accept

In [15]:
print("=" * 60)
print("Action 6: Auction")
print("=" * 60)
env = fresh_env()
# Give P0 2 containers on ship
st = env.state
env.state = st._replace(
    ship_contents=st.ship_contents.at[0, 0].set(1).at[0, 1].set(2))
print_state(env, "2 cargo at sea")
# Initiate
step_and_print(env, build_action(ACTION_MOVE_AUCTION), "initiate")
assert int(env.state.auction_active) == 1
# Bids (ACTION_MOVE_AUCTION + 1 = 8)
A = ACTION_MOVE_AUCTION + 1
step_and_print(env, jnp.array([A, 1, 0, 0, 5], dtype=jnp.int32), "P1 bids $5")
step_and_print(env, jnp.array([A, 2, 0, 0, 0], dtype=jnp.int32), "P2 bids $0")
step_and_print(env, jnp.array([A, 3, 0, 0, 2], dtype=jnp.int32), "P3 bids $2")
assert int(env.state.auction_round) == 1
# Seller accepts
step_and_print(env, jnp.array([A, 0, 0, 0, 1], dtype=jnp.int32), "P0 accepts")
assert int(env.state.auction_active) == 0
assert int(jnp.sum(env.state.island_store)) == 2
print("  ✓ passed")

Action 6: Auction
  [2 cargo at sea] P0 | cash=20 loans=0 factories=1 wh=1 harbour=0 ship=2 shop=0 prod=0 auction=0 acts=0
  → step(initiate): [np.int32(7), np.int32(1), np.int32(1), np.int32(1), np.int32(31)]
  [after initiate] P0 | cash=20 loans=0 factories=1 wh=1 harbour=0 ship=0 shop=0 prod=0 auction=1 acts=0
  → step(P1 bids $5): [np.int32(7), np.int32(1), np.int32(0), np.int32(0), np.int32(5)]
  [after P1 bids $5] P0 | cash=20 loans=0 factories=1 wh=1 harbour=0 ship=0 shop=0 prod=0 auction=1 acts=0
  → step(P2 bids $0): [np.int32(7), np.int32(2), np.int32(0), np.int32(0), np.int32(0)]
  [after P2 bids $0] P0 | cash=20 loans=0 factories=1 wh=1 harbour=0 ship=0 shop=0 prod=0 auction=1 acts=0
  → step(P3 bids $2): [np.int32(7), np.int32(3), np.int32(0), np.int32(0), np.int32(2)]
  [after P3 bids $2] P0 | cash=20 loans=0 factories=1 wh=1 harbour=0 ship=0 shop=0 prod=0 auction=1 acts=0
  → step(P0 accepts): [np.int32(7), np.int32(0), np.int32(0), np.int32(0), np.int32(1)]
  [after P0 

---
## Action 7 — Pass

Action: `[8, 0, 0, 0, P_STOP]`

In [16]:
print("=" * 60)
print("Action 7: Pass")
print("=" * 60)
env = fresh_env()
step_and_print(env, build_action(ACTION_PASS), "pass")
assert int(env.state.cash[0]) == INITIAL_CASH
print("  ✓ passed")

Action 7: Pass
  → step(pass): [np.int32(8), np.int32(1), np.int32(1), np.int32(1), np.int32(31)]
  [after pass] P0 | cash=20 loans=0 factories=1 wh=1 harbour=0 ship=0 shop=0 prod=0 auction=0 acts=1
  ✓ passed


---
## Action 8 — Take Loan

Does NOT consume an action.  Action: `[9, 0, 0, 0, P_STOP]`

In [17]:
print("=" * 60)
print("Action 8: Take Loan")
print("=" * 60)
env = fresh_env()
step_and_print(env, build_action(ACTION_TAKE_LOAN), "take loan")
assert int(env.state.loans[0]) == 1
assert int(env.state.cash[0]) == INITIAL_CASH + 10
print("  ✓ passed")

Action 8: Take Loan
  → step(take loan): [np.int32(9), np.int32(1), np.int32(1), np.int32(1), np.int32(31)]
  [after take loan] P0 | cash=30 loans=1 factories=1 wh=1 harbour=0 ship=0 shop=0 prod=0 auction=0 acts=0
  ✓ passed


---
## Action 9 — Repay Loan

Does NOT consume an action.  Action: `[10, 0, 0, 0, P_STOP]`

In [21]:
print("=" * 60)
print("Action 9: Repay Loan")
print("=" * 60)
env = fresh_env()
st = env.state
env.state = st._replace(loans=st.loans.at[0].set(1), cash=st.cash.at[0].set(40))
print_state(env, "1 loan, $40")
step_and_print(env, build_action(ACTION_REPAY_LOAN), "repay")
assert int(env.state.loans[0]) == 0
assert int(env.state.cash[0]) == 10
print("  ✓ passed")

Action 9: Repay Loan
  [1 loan, $40] P0 | cash=40 loans=1 factories=1 wh=1 harbour=0 ship=0 shop=0 prod=0 auction=0 acts=0
  → step(repay): [np.int32(10), np.int32(1), np.int32(1), np.int32(1), np.int32(31)]
  [after repay] P0 | cash=29 loans=0 factories=1 wh=1 harbour=0 ship=0 shop=0 prod=0 auction=0 acts=0


AssertionError: 

---
## Action 10 — Domestic Sale

Action: `[11, 0, colour+1, slot+1, P_STOP]`

In [ ]:
print("=" * 60)
print("Action 10: Domestic Sale")
print("=" * 60)
env = fresh_env()
# P0 has colour 0 at slot 4 ($5) in factory store
step_and_print(env, build_action(ACTION_DOMESTIC_SALE, color=0, price_slot=4),
               "sell colour 0 @ $5")
assert int(env.state.cash[0]) == INITIAL_CASH + 2
assert int(env.state.factory_store[0, 0, 4]) == 0
print("  ✓ passed")

---
## Summary

| Action | Steps | Active heads (per step) |
|--------|-------|------------------------|
| 0 Buy Factory | 1 | action_type, colour |
| 1 Buy Warehouse | 1 | action_type |
| 2 Produce | 1 + N | enter: action_type / recurrent: colour + price_slot |
| 3 Buy from Factory | 1 + N + STOP | step1: action_type + opponent / step2+: colour + purchase |
| 4 Move + Load | 1 + N + STOP | step1: action_type + opponent / step2+: colour + purchase |
| 5 Move Sea | 1 | action_type |
| 6 Auction | 1 + bids + decision | init: action_type / bid: action_type+opponent(bidder)+purchase(bid) |
| 7 Pass | 1 | action_type |
| 8/9 Take/Repay Loan | 1 | action_type |
| 10 Domestic Sale | 1 | action_type, colour, price_slot |

- **No-op at index 0** on every head — all meaningful values start at index 1.
- **Recurrent actions** split into entry + continuation.  During continuation,
  irrelevant heads are forced to no-op by action masks.
- **Shopping**: auto-cheapest source slot.  Harbour price via purchase (1-5).
- **Produce**: one step per factory (colour + price).
- **Auction**: opponent head = direct player index.  Purchase = bid amount.